# UVL Generation: Fine-tuned vs. Non-fine-tuned Model Comparison

This notebook compares the output of a **non-fine-tuned** (unsupervised) model against the **best performing fine-tuned** model (Qwen2.5-Coder-0.5B-Instruct, supervised) on the task of translating natural language feature descriptions into **Universal Variability Language (UVL)** code.


> **All outputs shown below are loaded directly from the evaluation result files** stored in `results/Supervised/` and `results/Unsupervised/`.

## Models compared

| Model | Training | Test Score (parsable) |
|-------|----------|----------------------|
| Qwen2.5-Coder-0.5B-Instruct | Unsupervised (no UVL examples) | 0 / 12 (0%) |
| **Qwen2.5-Coder-0.5B-Instruct** | **Supervised fine-tuning on UVL dataset** | **8 / 12 (67%)** |

The unsupervised model was only trained on raw UVL files for language-model pretraining and has never seen the translation task. The supervised model was fine-tuned with LoRA on (natural language → UVL) pairs.

In [4]:
import json
import pathlib

ROOT = pathlib.Path(".")  # adjust if running from a different directory
MODEL = "Qwen2.5-Coder-0.5B-Instruct"

sup_path   = ROOT / "results" / "Supervised"   / MODEL / "results.json"
unsup_path = ROOT / "results" / "Unsupervised" / MODEL / "results.json"

with open(sup_path)   as f: sup_results   = json.load(f)["test_results"]
with open(unsup_path) as f: unsup_results = json.load(f)["test_results"]

print(f"Loaded {len(sup_results)} supervised results   from: {sup_path}")
print(f"Loaded {len(unsup_results)} unsupervised results from: {unsup_path}")

Loaded 12 supervised results   from: results/Supervised/Qwen2.5-Coder-0.5B-Instruct/results.json
Loaded 12 unsupervised results from: results/Unsupervised/Qwen2.5-Coder-0.5B-Instruct/results.json


In [9]:
from IPython.display import display, HTML
import weasyprint

def strip_uvl_markers(text):
    text = text.strip()
    if text.startswith("'''uvl"):
        text = text[len("'''uvl"):]
    if text.endswith("'''"):
        text = text[:-3]
    return text.strip()

CSS = """
@page { size: 200mm auto; margin: 0; }
body { font-family: sans-serif; margin: 0; width: 200mm; }
.card { border: 1px solid #ddd; border-radius: 6px; overflow: hidden; margin: 12px 0; }
.header { background: #2c3e50; color: white; padding: 8px 14px; font-size: 13px; font-weight: bold; }
.badge { display: inline-block; padding: 1px 9px; border-radius: 4px; font-size: 11px; color: white; margin-left: 6px; }
.nl { background: #f5f5f5; padding: 10px 14px; border-bottom: 1px solid #ddd; font-size: 12px; }
.nl-label { font-size: 10px; color: #777; margin-bottom: 3px; }
.cols { display: flex; }
.col { flex: 1; padding: 10px 14px; }
.col-left { border-right: 1px solid #ddd; background: #fff8f8; }
.col-right { background: #f8fff8; }
.col-label { font-size: 10px; font-weight: bold; margin-bottom: 5px; }
pre { background: transparent; margin: 0; font-size: 10px; white-space: pre-wrap; word-break: break-word; }
.source { font-size: 9px; color: #999; padding: 4px 14px; background: #fafafa; border-top: 1px solid #eee; }
"""

def _build_html(index, difficulty, example_num):
    sup_entry    = sup_results[index]
    unsup_entry  = unsup_results[index]
    nl           = sup_entry["text"]
    sup_out      = strip_uvl_markers(sup_entry["output"])
    unsup_out    = unsup_entry["output"]
    sup_parsable = sup_entry["parsable"]
    diff_color   = {'Easy': '#27ae60', 'Medium': '#e67e22', 'Hard': '#c0392b'}.get(difficulty, '#555')
    parse_color  = '#2ecc71' if sup_parsable else '#e74c3c'
    parse_label  = '✔ parsable' if sup_parsable else '✘ not parsable'

    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><style>{CSS}</style></head><body>
<div class="card">
  <div class="header">
    Example {example_num}
    <span class="badge" style="background:{diff_color}">{difficulty}</span>
  </div>
  <div class="nl">
    <div class="nl-label">NATURAL LANGUAGE DESCRIPTION</div>
    {nl}
  </div>
  <div class="cols">
    <div class="col col-left">
      <div class="col-label" style="color:#c0392b">
        ✘ NON-FINE-TUNED
        <span class="badge" style="background:#e74c3c">not parsable</span>
      </div>
      <pre>{unsup_out}</pre>
    </div>
    <div class="col col-right">
      <div class="col-label" style="color:#27ae60">
        ✔ FINE-TUNED (Supervised — {MODEL})
        <span class="badge" style="background:{parse_color}">{parse_label}</span>
      </div>
      <pre>{sup_out}</pre>
    </div>
  </div>
  <div class="source">
    source: test_results[{index}] — {sup_path} / {unsup_path}
  </div>
</div>
</body></html>"""

def show_comparison(index, difficulty, example_num, save_pdf=None):
    html = _build_html(index, difficulty, example_num)
    display(HTML(html))
    if save_pdf:
        weasyprint.HTML(string=html).write_pdf(save_pdf)
        print(f"Saved: {save_pdf}")

---
## Example 1 — Easy

In [10]:
# test_results[0]: car configuration (Easy)
show_comparison(index=0, difficulty="Easy", example_num=1, save_pdf="results/example1_easy.pdf")

Saved: results/example1_easy.pdf


---
## Example 2 — Medium

In [11]:
# test_results[5]: chocolate production line (Medium)
show_comparison(index=5, difficulty="Medium", example_num=2, save_pdf="results/example2_medium.pdf")

Saved: results/example2_medium.pdf


---
## Example 3 — Hard

In [12]:
# test_results[10]: CI/CD deployment pipelines (Hard)
show_comparison(index=10, difficulty="Hard", example_num=3, save_pdf="results/example3_hard.pdf")

Saved: results/example3_hard.pdf
